# Linear Algebra + ODEs + Z-Transform in SymPy
KSP orbital mechanics, LoL/Dota2 projectiles, GS phase recovery.

In [ ]:
import sympy as sp
import numpy as np
sp.init_printing(use_unicode=False, wrap_line=False)
print('imports OK')

## S1. Linear Algebra

In [ ]:
A = sp.Matrix([[2, 1],[1, 3]])
print('A ='); sp.pprint(A)
print('det(A) =', A.det())
print('inv(A) ='); sp.pprint(A.inv())
print('A*inv(A) ='); sp.pprint(sp.simplify(A * A.inv()))

In [ ]:
print('Eigenvalues/eigenvectors of A:')
for val, mult, vecs in A.eigenvects():
    print(f'  lambda={val}  mult={mult}')
    for v in vecs:
        sp.pprint(v.T)

In [ ]:
# LoL damage system: solve for armor_pen and AP
A_sys = sp.Matrix([[3, 1],[1, 2]])
b_sys = sp.Matrix([1000, 1000])
x_sol = A_sys.solve(b_sys)
print('LoL damage system Ax=b:')
print(f'  x1 (armor pen) = {x_sol[0]}')
print(f'  x2 (AP)        = {x_sol[1]}')
print(f'  Check 3*x1+x2  = {3*x_sol[0]+x_sol[1]}')

## S2. ODEs -- dsolve

In [ ]:
t = sp.Symbol('t', positive=True)
y = sp.Function('y')
tau, V0 = sp.symbols('tau V0', positive=True)

# RC circuit / DRAM leakage: dy/dt = -y/tau
ode1 = sp.Eq(y(t).diff(t), -y(t)/tau)
sol1 = sp.dsolve(ode1, y(t))
print('RC: dy/dt = -y/tau'); sp.pprint(sol1)

C1_val = sp.solve(sol1.rhs.subs(t, 0) - V0, 'C1')[0]
sol1_iv = sol1.subs('C1', C1_val)
print('y(0)=V0:'); sp.pprint(sol1_iv)

In [ ]:
# Spring-mass (KSP landing strut): y'' + 2*zeta*omega*y' + omega^2*y = 0
omega, zeta = sp.symbols('omega zeta', positive=True)
ode2 = sp.Eq(y(t).diff(t,2) + 2*zeta*omega*y(t).diff(t) + omega**2*y(t), 0)
sol2 = sp.dsolve(ode2, y(t))
print('KSP landing strut (spring-mass):'); sp.pprint(sol2)

print('Roots for omega=1:')
for name, z_val in [('underdamped',0.3),('critical',1.0),('overdamped',2.0)]:
    r = np.roots([1, 2*z_val, 1])
    print(f'  {name:15s} zeta={z_val}  roots={r[0]:.3f}, {r[1]:.3f}')

In [ ]:
# KSP circular orbit
r_orb, G, M_k = sp.symbols('r G M', positive=True)
omega_orb = sp.sqrt(G*M_k / r_orb**3)
T_orb = sp.simplify(2*sp.pi / omega_orb)
print('Kepler 3rd law: T ='); sp.pprint(T_orb)

G_n=6.674e-11; M_Kerbin=5.2915e22; R_Kerbin=600e3; alt=80e3
r_LKO = R_Kerbin+alt
T_LKO = 2*np.pi*np.sqrt(r_LKO**3/(G_n*M_Kerbin))
v_LKO = 2*np.pi*r_LKO/T_LKO
print(f'Low Kerbin Orbit: T={T_LKO/60:.1f} min  v={v_LKO:.0f} m/s')
print(f'Compare Earth LEO: ~90 min, ~7800 m/s')

In [ ]:
# Projectile: LoL/Dota2 skillshot
g_s, v0, theta_s = sp.symbols('g v0 theta', positive=True)
tp = sp.Symbol('t', positive=True)
y_t = v0*sp.sin(theta_s)*tp - g_s/2*tp**2
t_flight = sp.solve(y_t, tp)[0]
x_t = v0*sp.cos(theta_s)*tp
x_range = sp.simplify(x_t.subs(tp, t_flight))
print('Projectile time of flight:'); sp.pprint(t_flight)
print('Range:'); sp.pprint(x_range)
print('Max range at theta=pi/4 (45 deg)')

v0_n=3000.0; g_n=9.8
for deg in [30,45,60]:
    th=np.radians(deg)
    R=v0_n**2*np.sin(2*th)/g_n
    print(f'  theta={deg} deg  range={R:.0f} Dota units ({R*0.00075:.2f} km)')

## S3. Z-Transform

In [ ]:
print('Z-transform pairs:')
pairs = [
    ('delta[n]',     '1',           'impulse'),
    ('u[n]',         'z/(z-1)',     'unit step'),
    ('a^n u[n]',     'z/(z-a)',     'causal exponential'),
    ('n u[n]',       'z/(z-1)^2',  'ramp'),
]
for xn, Xz, name in pairs:
    print(f'  {xn:15s} <-> {Xz:20s} [{name}]')
print()
print('z = exp(j*omega) on unit circle = DTFT')
print('z = exp(s*T) connects s-domain (Laplace) to z-domain')

In [ ]:
z_s = sp.Symbol('z')

# IIR filter H(z) = z/(z - 1/2), pole at z=0.5
H_iir = z_s / (z_s - sp.Rational(1,2))
print('IIR H(z) = z/(z-1/2):')
print('  Partial fractions:'); sp.pprint(sp.apart(H_iir, z_s))

h = np.array([0.5**n for n in range(16)])
print(f'  h[n] = (0.5)^n, sum={h.sum():.4f} (-> 2.0 = 1/(1-0.5))')

In [ ]:
# Bilinear transform (Tustin): s = (2/T)*(z-1)/(z+1)
T_sym = sp.Symbol('T', positive=True)
omega_c = sp.Symbol('omega_c', positive=True)
bil = sp.Rational(2)*( z_s-1 ) / ( T_sym*(z_s+1) )
z_pole = sp.solve(bil - (-omega_c), z_s)[0]
print('Bilinear: analog pole s=-omega_c maps to digital z=')
sp.pprint(sp.simplify(z_pole))
print('For T*omega_c << 1: z_pole ~ 1 - T*omega_c (near 1, stable)')

In [ ]:
# GS: H(nu) = exp(i*pi*D*nu^2) is all-pass -> on unit circle
D_val=5000; N=512
nu = np.fft.fftfreq(N)
H = np.exp(1j*np.pi*D_val*nu**2)
print('GS dispersion filter:')
print(f'  |H(nu)| range: [{np.abs(H).min():.6f}, {np.abs(H).max():.6f}]')
print('  All-pass: purely on unit circle in Z-domain')
print('  One measurement gives |H*E|=|E|: phase is invisible')
print('  Two measurements (D1!=D2): triangulate phase -> GS works')

## S4. if/but -- conditional logic

| English | Math | Python | C |
|---------|------|--------|---|
| if A then B | A => B | `if A: B` | `if(A){B;}` |
| A but not B | A AND NOT B | `A and not B` | `A && !B` |
| A xor B | A XOR B | `A ^ B` | `A ^ B` |
| for all n | forall n | `all(...)` | for+assert |
| there exists | exists n | `any(...)` | for+break |

In [ ]:
x = sp.Symbol('x', real=True)

# Piecewise (math if/else)
rect = sp.Piecewise((1, sp.Abs(x) <= sp.Rational(1,2)), (0, True))
print('rect(x):')
for val in [-1, -0.5, 0, 0.5, 1]:
    print(f'  rect({val}) = {rect.subs(x, val)}')

# LoL armor damage reduction
armor, raw = sp.symbols('armor raw', real=True)
eff = sp.Piecewise(
    (raw * 100/(100+armor),            armor >= 0),
    (raw * (2 - 100/(100-armor)),      armor < 0)
)
print()
print('LoL effective damage (100 raw):')
for a in [-20, 0, 50, 100, 200]:
    print(f'  armor={a:4d}  eff={float(eff.subs([(armor,a),(raw,100)])):.1f}')